In [ ]:
"""
================================================================================
  Qwen-0.5B × Amharic Fine-Tuning Demo - FINAL WORKING VERSION
  Using: addisai/FineTome-single-turn-dedup-amharic (83k examples)
================================================================================
"""

# Install correct versions
!pip uninstall -y peft transformers accelerate -q
!pip install transformers==4.41.2 accelerate==0.31.0 peft==0.11.0 datasets sentencepiece -q

import os
import time
import warnings
import torch

warnings.filterwarnings("ignore")

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

# Check GPU
print("=" * 60)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    # Check if bfloat16 is supported
    if torch.cuda.is_bf16_supported():
        print("✓ bfloat16 supported")
        dtype = torch.bfloat16
    else:
        print("✓ Using float16")
        dtype = torch.float16

# ── 1. Constants ──────────────────────────────────────────────────────────────
MODEL_NAME     = "Qwen/Qwen2-0.5B-Instruct"
DATASET_NAME   = "addisai/FineTome-single-turn-dedup-amharic"
OUTPUT_DIR     = "./qwen-amharic-lora-improved"
MAX_SEQ_LEN    = 512
LORA_RANK      = 16
LORA_ALPHA     = 32
TRAIN_EPOCHS   = 2
BATCH_SIZE     = 2  # Reduced for stability
GRADIENT_ACCUMULATION = 4  # Effective batch size = 2 * 4 = 8
LR             = 2e-4


# ── 2. Tokenizer ──────────────────────────────────────────────────────────────
print("=" * 60)
print("Loading tokenizer …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"✓ Tokenizer loaded")


# ── 3. Base model ────────────────────────────────────────────────────────────
print("Loading base model on GPU …")
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,  # Use bfloat16 instead of float16
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print(f"✓ Model loaded in {time.time()-t0:.1f}s")

if torch.cuda.is_available():
    used = torch.cuda.memory_allocated(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"💾 Memory used: {used:.2f} GB / {total:.1f} GB ({used/total*100:.1f}%)")


# ── 4. BEFORE fine-tune: baseline responses ───────────────────────────────────
TEST_PROMPTS_AM = [
    "ስለ ምርታችሁ የዋጋ ማወቅ እፈልጋለሁ።",
    "ትዕዛዜን እንዴት ልከታተለው እችላለሁ?",
    "ምርቱን መመለስ ከፈለኩ ምን ማድረግ አለብኝ?",
]

def chat(model_obj, user_msg, max_new_tokens=150, label=""):
    messages = [
        {"role": "system", "content": "You are a helpful Ethiopian customer service assistant. Respond in Amharic or English as appropriate."},
        {"role": "user", "content": user_msg},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = tokenizer(text, return_tensors="pt")
    device = next(model_obj.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model_obj.generate(
            inputs["input_ids"],
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n[{label}] User: {user_msg}")
    print(f"[{label}] Reply: {response[:250]}")
    return response


print("\n" + "=" * 60)
print("BEFORE FINE-TUNE — Baseline responses")
print("=" * 60)
before_responses = []
for prompt in TEST_PROMPTS_AM:
    try:
        resp = chat(model, prompt, label="BEFORE")
        before_responses.append(resp)
    except Exception as e:
        print(f"Error: {e}")
        before_responses.append("")
    time.sleep(1)


# ── 5. Load FineTome Amharic Dataset ──────────────────────────────────────────
print("\n" + "=" * 60)
print(f"Loading dataset: {DATASET_NAME} …")
raw = load_dataset(DATASET_NAME, split="train")
print(f"✓ Loaded {len(raw)} examples")

def format_conversation_to_instruction(ex):
    """Convert FineTome's conversation format to instruction tuning format."""
    conv_am = ex.get("conversations_amharic", [])

    if conv_am and len(conv_am) >= 2:
        user_msg = conv_am[0].get("content", "") if len(conv_am) > 0 else ""
        assistant_msg = conv_am[1].get("content", "") if len(conv_am) > 1 else ""

        instruction = f"Customer question: {user_msg}"
        output = assistant_msg
    else:
        conv_en = ex.get("conversations", [])
        if conv_en and len(conv_en) >= 2:
            user_msg = conv_en[0].get("content", "")
            assistant_msg = conv_en[1].get("content", "")
            instruction = f"Customer question: {user_msg}"
            output = assistant_msg
        else:
            instruction = "Provide helpful customer service"
            output = "I'm here to help you with your request."

    text = f"### Instruction:\n{instruction}\n\n### Response:\n{output}{tokenizer.eos_token}"
    return {"text": text}

print("\nFormatting dataset for training...")
dataset = raw.map(format_conversation_to_instruction, remove_columns=raw.column_names)

NUM_EXAMPLES = 2000  # Reduced for faster training
dataset = dataset.select(range(min(NUM_EXAMPLES, len(dataset))))
print(f"Using {len(dataset)} examples for training")

def tokenize(ex):
    tok = tokenizer(
        ex["text"],
        max_length=MAX_SEQ_LEN,
        truncation=True,
        padding="max_length",
    )
    tok["labels"] = tok["input_ids"].copy()
    return tok

print("Tokenizing dataset...")
tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
split = tokenized.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print(f"Train: {len(train_ds)}, Eval: {len(eval_ds)}")


# ── 6. LoRA Configuration ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Setting up LoRA...")
print("=" * 60)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
)

model.train()
model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Trainable parameters: {trainable_params:,} ({trainable_params/total_params:.2%} of {total_params:,})")


# ── 7. Training (Fixed gradient scaling issue) ────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    warmup_steps=50,
    learning_rate=LR,
    bf16=True,  # Use bfloat16 instead of fp16 (more stable)
    logging_steps=20,
    evaluation_strategy="steps",
    eval_steps=50,
    save_strategy="no",
    report_to="none",
    dataloader_num_workers=0,
    max_grad_norm=0.3,  # Add gradient clipping for stability
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True),
)

print("\n" + "=" * 60)
print("Starting Enhanced LoRA Fine-tuning …")
print("=" * 60)
print(f"⚠️ Training on {len(train_ds)} examples")
print(f"⚠️ {TRAIN_EPOCHS} epochs")
print(f"⚠️ Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"⚠️ Estimated time: 10-15 minutes on Colab GPU\n")

t_train = time.time()
trainer.train()
print(f"\n✓ Training completed in {(time.time()-t_train)/60:.1f} minutes")


# ── 8. Save model ─────────────────────────────────────────────────────────────
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✓ Model saved to {OUTPUT_DIR}")


# ── 9. AFTER fine-tune: compare responses ─────────────────────────────────────
model.eval()

print("\n" + "=" * 60)
print("AFTER FINE-TUNE — Improved responses (with FineTome dataset)")
print("=" * 60)
after_responses = []
for prompt in TEST_PROMPTS_AM:
    try:
        resp = chat(model, prompt, label="AFTER")
        after_responses.append(resp)
    except Exception as e:
        print(f"Error: {e}")
        after_responses.append("")
    time.sleep(1)


# ── 10. Final Comparison ──────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("📊 COMPARISON: BEFORE vs AFTER Fine-Tuning")
print("=" * 60)

for i, prompt in enumerate(TEST_PROMPTS_AM):
    print(f"\n{'='*60}")
    print(f"Prompt {i+1}: {prompt}")
    print(f"{'='*60}")
    print(f"🔴 BEFORE: {before_responses[i][:200]}" if before_responses[i] else "🔴 BEFORE: [No response]")
    print(f"🟢 AFTER:  {after_responses[i][:200]}" if after_responses[i] else "🟢 AFTER: [No response]")
    print()

print("\n" + "=" * 60)
print("✅ ENHANCED DEMONSTRATION COMPLETE!")
print("=" * 60)
print(f"""
IMPROVEMENTS:
├─ Dataset: {len(train_ds)} examples
├─ LoRA Rank: {LORA_RANK}
├─ Trainable params: {trainable_params:,} ({trainable_params/total_params:.2%})
├─ Using bfloat16 (more stable than fp16)
└─ Gradient clipping for stability

Training completed in {(time.time()-t_train)/60:.1f} minutes
Final eval loss: {trainer.state.log_history[-2]['eval_loss']:.4f}
""")

# Memory cleanup
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"Final GPU memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

CUDA available: True
GPU: Tesla T4
GPU Memory: 14.6 GB
✓ bfloat16 supported
Loading tokenizer …


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


✓ Tokenizer loaded
Loading base model on GPU …
✓ Model loaded in 1.9s
💾 Memory used: 2.26 GB / 14.6 GB (15.5%)

BEFORE FINE-TUNE — Baseline responses

[BEFORE] User: ስለ ምርታችሁ የዋጋ ማወቅ እፈልጋለሁ።
[BEFORE] Reply: "Ẹ ṣe fẹ ki n kọ mọ, mi ni mo le tọ? A sii pe sii pe ki n gẹẹni."

[BEFORE] User: ትዕዛዜን እንዴት ልከታተለው እችላለሁ?
[BEFORE] Reply: "Tetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetetet

[BEFORE] User: ምርቱን መመለስ ከፈለኩ ምን ማድረግ አለብኝ?
[BEFORE] Reply: ምርት ዓልጠራዕ ለማርኛ ታሔ缟 ለረፀ ይታችቸ ህርታ ነው ያም ስም ዘይም ይን ብነት ይል ይራ ተላም ሴደረታ ለገር ቢበል ሚፍበባ ሄራ ተለው ዲህ ለረፀ ይያ ጎበል ነው በይ ይህ ህረታ �

Loading dataset: addisai/FineTome-single-turn-dedup-amharic …
✓ Loaded 83290 examples

Formatting dataset for training...


Map:   0%|          | 0/83290 [00:00<?, ? examples/s]

Using 2000 examples for training
Tokenizing dataset...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Train: 1800, Eval: 200

Setting up LoRA...
✓ Trainable parameters: 1,081,344 (0.22% of 495,114,112)

Starting Enhanced LoRA Fine-tuning …
⚠️ Training on 1800 examples
⚠️ 2 epochs
⚠️ Effective batch size: 8
⚠️ Estimated time: 10-15 minutes on Colab GPU



Step,Training Loss,Validation Loss
